# 順序回帰（Ordinal GP Regression）とActive Learning：初学者向け実践ガイド

このNotebookでは、製品品質などの **順序がある3クラス（0: 不良・1: 要確認・2: 良好）** を対象に、
**学習 → 予測確率の解釈 → Active Learningで候補提案 → ラベルの取得 → 再学習** を実行します。
最後にカテゴリ入力を持つMixed Ordinal GPと、入力制約付きの候補探索も試します。

通常のマルチクラスGPと違い、Ordinal GPは**1つの潜在関数と単調な閾値**で順序を表します。
「クラス0と1の差」と「クラス1と2の差」が等しいとは仮定しません。

関連資料：[理論（数式の導出）](../../docs/theory/31_ordinal_classification.md) /
[実装との対応](../../docs/guide/31_ordinal_classification.md) /
[分類モデル一覧](../../docs/models/classification.md)


**回帰目的との組み合わせ：** 順序分類を制約として使い、単目的・多目的回帰の実験候補を提案する例は[61番：異種モデルによる分類制約付きBO](61_heterogeneous_classification_constraints.ipynb)にまとめています。


**独立した理論・実装ガイド：** [順序GP分類の理論（第31章）](../../docs/theory/31_ordinal_classification.md) / [順序GP分類ガイド](../../docs/guide/31_ordinal_classification.md)。二値分類の基礎は[第24章](../../docs/theory/24_classification.md)を参照してください。


## 0. 全体の流れ

1. 初期の観測データ `X, y`（ラベルは0、1、2）を用意する
2. `OrdinalSingleTaskGPClassifier` を学習する
3. `posterior(X)` と `probability_posterior(X)` の違い、信用区間を確認する
4. Entropy・BALDで「次にラベルを測りたい条件」を評価する
5. q=2の候補をまとめて提案する
6. Mixed（連続＋カテゴリ）入力も試す
7. 未観測候補から選択→観測→再学習を**2回**繰り返す
8. 保留データで評価指標を確認する

**重要：** 実験条件を提案するだけではラベルは得られません。
実際には検査・実験・専門家判定でラベルを取得します。
ここでは理解のために**人工的なラベル生成規則**を検査装置の代わりに使います。
実データと比較するための性能実証ではありません。


### 候補最適化APIの使い分け

| 探索する入力 | 使うAPI | 理由 |
|---|---|---|
| 連続2変数・q=2 | `SobolSearchStrategy.optimize()` | 分類獲得関数をSobol候補から最大化 |
| Mixed（連続＋設備カテゴリ）・q=2・入力制約付き | `optimize_acqf_sampling()` | `MixedVariableSpace` と `CandidateConstraints` を同時に扱う既存の探索API |
| 既知の未観測候補プール・q=1 | BoTorch `optimize_acqf_discrete()` | 有限候補から獲得関数を最大化し、選んだ候補をプールから除外 |

いずれも**予測クラス番号を最大化する処理ではありません**。Entropyなどの分類用獲得関数を最適化します。

## 1. 初期データ：入力 `X` と順序ラベル `y`

2つの連続入力条件を使います。第一列は品質に主な影響を与える条件、
第二列は補助的な条件と想定します。
説明のため、隠れた品質スコア `x0 + 0.12*x1` を
0.4と0.75で区切って3つの順序ラベルを作ります。

**ラベルは観測値としてモデルへ渡します。**
実際のモデルが、この人工規則や境界値0.4・0.75を知っているわけではありません。


In [ ]:
import torch

from robotorchan.acquisition.active_learning.classification import (
    BALD,
    MarginUncertainty,
    PredictiveEntropy,
    qBALD,
    qPredictiveEntropy,
)
from robotorchan.models.classification import (
    MixedOrdinalSingleTaskGPClassifier,
    OrdinalSingleTaskGPClassifier,
)
from robotorchan.optim.backends.sampling import optimize_acqf_sampling
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.optim.strategies.sobol import SobolSearchStrategy

torch.manual_seed(42)
dtype = torch.double
thresholds_for_synthetic_labels = torch.tensor([0.4, 0.75], dtype=dtype)

condition = torch.linspace(0.05, 0.95, 18, dtype=dtype)
auxiliary = 0.5 + 0.35 * torch.sin(6.0 * condition)
train_X = torch.stack((condition, auxiliary), dim=-1)
synthetic_quality = train_X[:, 0] + 0.12 * train_X[:, 1]
train_Y = torch.bucketize(
    synthetic_quality.contiguous(), thresholds_for_synthetic_labels
)
print("入力形状:", train_X.shape, "ラベル形状:", train_Y.shape)
print("実測扱いのクラス:", train_Y.unique(sorted=True))
assert train_X.shape == (18, 2)
assert torch.equal(train_Y.unique(), torch.arange(3))


## 2. 単一潜在GPで順序ラベルを学習する

`num_classes=3` は順序ラベル0、1、2の3段階を意味します。
`inducing_points=8` は変分GPの計算を近似する代表点の数であり、
追加の観測データ8点を意味しません。

`fit()` はGPカーネル・変分分布と、**Ordered-probitの閾値**を学習します。
このNotebookではCIで動かしやすいように学習回数を小さくしています。
実務では検証データで学習回数・学習率・誘導点数を調整してください。


In [ ]:
model = OrdinalSingleTaskGPClassifier(
    train_X, train_Y, num_classes=3, inducing_points=8
)
model.fit(epochs=4, lr=0.03)
learned_thresholds = model.likelihood.thresholds.detach()
print("学習した閾値:", learned_thresholds)
print("クラスの並び:", model.class_labels)
assert learned_thresholds.shape == (2,)
assert bool((learned_thresholds[1:] > learned_thresholds[:-1]).all())


## 3. 潜在値・閾値・予測確率の違い

Ordinal GPは**クラス数個の潜在GP**を作りません。
一つの潜在GPから得た連続値 `f(x)` と、順序を保つ閾値を用います。

- `posterior(X)`：潜在関数のGaussianな事後分布。平均の最終次元は**1**
- `predict_proba(X)`：各クラスへの**事後予測確率**。最終次元は**3**
- `probability_posterior(X)`：クラス確率そのものの事後モーメントと周辺信用区間
- `predict_class(X)`：確率が最大のクラス番号
- `predictive_entropy(X)`：予測ラベルがどれほど曖昧かを表す値

閾値を潜在平均に適用するだけでなく、
**潜在GPの分散を含めて積分**することで予測確率を計算します。
詳細は[理論第24章](../../docs/theory/24_classification.md)にあります。


In [ ]:
test_X = torch.tensor(
    [[0.2, 0.5], [0.5, 0.5], [0.8, 0.5]], dtype=dtype
)
latent = model.posterior(test_X)
probabilities = model.predict_proba(test_X)
predicted_classes = model.predict_class(test_X)
entropy = model.predictive_entropy(test_X)

print("潜在平均（確率ではない）:", latent.mean.squeeze(-1))
print("クラス0・1・2の予測確率:\n", probabilities)
print("予測ラベル:", predicted_classes)
print("ラベルのEntropy:", entropy)
assert latent.mean.shape == (3, 1)
assert probabilities.shape == (3, 3)
torch.testing.assert_close(probabilities.sum(-1), torch.ones(3, dtype=dtype))


### 3.1 Ordinal確率Posteriorと信用区間を可視化する

Ordinal GPは潜在関数が1次元でも、`probability_posterior(X).mean` の最後の軸はクラス数3です。
`mean` はOrdered-probitをガウス積分した解析的な予測確率、
`variance` は**クラス確率そのもの**の事後分散をGauss–Hermite求積で近似した値です。

`credible_interval(0.90)` は、潜在事後からサンプリングした
各クラス確率の5%点〜95%点を取ります。
特に**中間クラスの確率は潜在関数に対して単調ではない**ため、
二値Probitと同じ潜在Gaussianの上下分位点の変換では正しい区間を求められません。
この帯はラベル0・1・2の予測集合でも、同時90%信用領域でもありません。


In [ ]:
import matplotlib.pyplot as plt

# 第2条件を固定し、第一条件に沿ってクラス0・1・2を比較します。
slice_x = torch.linspace(0.05, 0.95, 41, dtype=dtype)
slice_X = torch.stack((slice_x, torch.full_like(slice_x, 0.5)), dim=-1)
with torch.no_grad():
    probability_posterior = model.probability_posterior(slice_X)
    class_mean = probability_posterior.mean
    class_variance = probability_posterior.variance
    class_lower, class_upper = probability_posterior.credible_interval(
        0.90, num_samples=512
    )
    class_samples = probability_posterior.rsample(torch.Size([8]))
    torch.testing.assert_close(class_mean, model.predict_proba(slice_X))

assert class_mean.shape == class_variance.shape == (41, 3)
assert class_lower.shape == class_upper.shape == (41, 3)
torch.testing.assert_close(class_mean.sum(-1), torch.ones(41, dtype=dtype))
torch.testing.assert_close(class_samples.sum(-1), torch.ones_like(class_samples[..., 0]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for cls in range(model.num_classes):
    x = slice_x.numpy()
    axes[0].plot(x, class_mean[:, cls].numpy(), label=f"Class {cls} mean")
    axes[0].fill_between(
        x, class_lower[:, cls].numpy(), class_upper[:, cls].numpy(), alpha=0.15
    )
    axes[1].plot(x, class_variance[:, cls].numpy(), label=f"Class {cls} variance")
axes[0].set_ylabel("Class probability")
axes[1].set_ylabel("Epistemic probability variance")
axes[0].set_ylim(-0.05, 1.05)
for ax in axes:
    ax.set_xlabel("First condition (second = 0.5)")
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# q=2でも、候補次元とクラス次元をそれぞれ維持します。
posterior_q2 = model.probability_posterior(slice_X[:2].unsqueeze(0))
assert posterior_q2.mean.shape == (1, 2, 3)


### 条件付きクラス確率と平均予測確率は違う

`sample_class_probabilities` は潜在GPから複数の値をサンプルし、
それぞれをOrdered-probitの条件付き確率へ変換します。
その**サンプル間のばらつき**がモデル由来の不確実性を表します。
一方 `predict_proba` は潜在Gaussian分布を解析的に周辺化した
**予測ラベルの確率**です。

両者が似た値になっても、前者を単純に複製したものが後者ではありません。


In [ ]:
probability_samples = model.sample_class_probabilities(
    test_X, sample_shape=torch.Size([32])
)
print("確率サンプルの形状:", probability_samples.shape)
print("最初の入力に対する確率サンプルの平均:", probability_samples[:, 0].mean(0))
print("解析的な予測確率:", probabilities[0])
assert probability_samples.shape == (32, 3, 3)


## 4. Active Learning：次に測る条件を比較する

未測定の候補を並べ、獲得関数で点数を付けます。

- **PredictiveEntropy**：ラベルが曖昧な条件を選ぶ
- **MarginUncertainty**：上位2クラスの予測確率差が小さい条件を選ぶ
- **BALD**：追加観測で潜在モデルについて情報が増えそうな条件を選ぶ

`BALD` は不確実性をモンテカルロサンプルから近似するので、
乱数によって点数が多少変わります。
**スコアが大きいことは、品質が高いことを意味しません。**


In [ ]:
candidate_x0 = torch.linspace(0.05, 0.95, 12, dtype=dtype)
candidate_X = torch.stack(
    (candidate_x0, torch.full_like(candidate_x0, 0.5)), dim=-1
)
# 単一点の獲得関数には [候補数, q=1, 入力次元] を渡します。
candidate_batches = candidate_X.unsqueeze(-2)
entropy_scores = PredictiveEntropy(model)(candidate_batches)
margin_scores = MarginUncertainty(model)(candidate_batches)
bald_scores = BALD(model, num_samples=16)(candidate_batches)
print("Entropy候補:", candidate_X[entropy_scores.argmax()])
print("Margin候補:", candidate_X[margin_scores.argmax()])
print("BALD候補:", candidate_X[bald_scores.argmax()])
assert entropy_scores.shape == margin_scores.shape == bald_scores.shape == (12,)
assert torch.isfinite(bald_scores).all()


### 単一候補でも最適化関数を使う

上のセルではEntropy・Margin・BALDのスコアを比較しました。次のセルでは、同じ合法な候補集合に対してBoTorch公開 `optimize_acqf_discrete` を使い、Entropyが最大の1点を選びます。

これは候補集合を与える離散最適化です。全連続空間の厳密最適化ではありませんが、実験可能な条件が列挙されている場合に自然な方法です。

In [ ]:
from botorch.optim import optimize_acqf_discrete

single_candidate, single_value = optimize_acqf_discrete(
    acq_function=PredictiveEntropy(model),
    q=1,
    choices=candidate_X,
    unique=True,
)
print("最適化で提案した単一候補:", single_candidate)
print("Entropy獲得関数値:", single_value)
assert single_candidate.shape == (1, 2)

## 4.2 順序回帰としての最大化：既存の `PosteriorMean` を使う

Active LearningのEntropy最大化は**ラベルが不確かな点**を探します。一方、順序回帰を使った品質改善では、**良い品質につながりそうな条件**を選びたい場合があります。

Ordered-probitでは潜在関数 `f(x)` が大きいほど、より上位の順序クラスが出やすくなります。そこでBoTorch既存の `PosteriorMean(model)` を使って、潜在GPの事後平均を最大化します。これは**潜在スコアの活用（exploitation）**であり、Ordinal GPの確率や改善量を直接最大化する専用BO獲得関数ではありません。

さらに `predict_proba()` から**期待効用**と**良好クラス確率**を計算し、同じ候補プールで比較します。効用 `[0,1,2]` は意思決定上の仮定であり、クラス間の距離が等しいとモデルが仮定しているわけではありません。

In [ ]:
from botorch.acquisition.analytic import PosteriorMean

quality_choices = torch.stack((
    torch.linspace(0.0, 1.0, 51, dtype=dtype),
    torch.full((51,), 0.5, dtype=dtype),
), dim=-1)

# 既存のBoTorch獲得関数でOrdinal GPの単一潜在平均を最大化
latent_mean_acq = PosteriorMean(model)
latent_best_X, latent_best_value = optimize_acqf_discrete(
    acq_function=latent_mean_acq, q=1, choices=quality_choices, unique=True
)

# 期待効用と良好クラス確率は公開予測APIから直接算出する
utilities = torch.tensor([0.0, 1.0, 2.0], dtype=dtype)
with torch.no_grad():
    quality_probabilities = model.predict_proba(quality_choices)
    expected_quality = quality_probabilities @ utilities
    good_class_probability = quality_probabilities[:, 2]
    quality_entropy = PredictiveEntropy(model)(quality_choices.unsqueeze(-2))
expected_best_X = quality_choices[expected_quality.argmax()].unsqueeze(0)
entropy_best_X, entropy_value = optimize_acqf_discrete(
    acq_function=PredictiveEntropy(model), q=1, choices=quality_choices, unique=True
)
print("潜在平均最大の条件:", latent_best_X, "latent mean:", latent_best_value)
print("期待効用最大の条件:", expected_best_X)
print("Entropy最大の条件:", entropy_best_X, "entropy:", entropy_value)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
axes[0].plot(quality_choices[:, 0], expected_quality.detach(), label="Expected ordinal utility")
axes[0].axvline(latent_best_X[0, 0].item(), color="tab:green", linestyle="--", label="maximize latent mean")
axes[0].set(ylabel="Expected utility", title="Optimization: seek better quality")
axes[1].plot(quality_choices[:, 0], quality_entropy.detach(), label="Predictive entropy")
axes[1].axvline(entropy_best_X[0, 0].item(), color="tab:orange", linestyle="--", label="query uncertain label")
axes[1].set(ylabel="Entropy", title="Active Learning: seek information")
for ax in axes:
    ax.set_xlabel("First condition (second = 0.5)")
    ax.legend()
plt.tight_layout()
plt.show()

### 「良好クラスになる確率」を直接比較する

製品仕様が「良好（クラス2）なら合格」であれば `P(Y=2|x)` を最大化することもできます。これは `predict_proba()` から直接得られるので、Notebook専用の獲得関数クラスを定義する必要はありません。

ただし、**潜在平均・期待効用・良好クラス確率は異なる目的**です。事後分散が入力によって異なる場合、最大化する条件も一致するとは限りません。今回の期待効用と良好クラス確率の最大値は有限候補プール上の `argmax` で求めます。これらを `optimize_acqf_discrete` に渡すには別途獲得関数のラッパーが必要になるため、既存APIだけで実現できる範囲に留めます。

In [ ]:
good_best_X = quality_choices[good_class_probability.argmax()].unsqueeze(0)
print("良好クラス確率最大の条件:", good_best_X)
print("P(Y=2 | x):", good_class_probability.max().item())
print("潜在平均最大の条件と同じとは限りません")

## 5. q=2：2つの条件をまとめて提案する

`qPredictiveEntropy` は2候補の周辺Entropyを平均します。
**候補間の重複情報を直接抑制する関数ではありません。**

`qBALD` は潜在GPの同時サンプルから候補間の依存を考慮します。
ただしラベルベクトルの組合せ数が `C**q <= 256` である必要があります。
この例は3クラス・2候補なので `3**2=9` 通りです。

`SobolSearchStrategy` は既存の候補生成器で、**獲得関数が高い条件**を提案します。
提案結果にはクラスラベルではなく入力条件が入ります。


In [ ]:
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype)
strategy = SobolSearchStrategy(bounds, num_samples=16, seed=17)
result = strategy.optimize(qPredictiveEntropy(model), q=2)
print("提案された2つの連続条件:\n", result.candidates)
print("周辺Entropy平均:", result.acquisition_value)
joint_bald_score = qBALD(model, num_samples=16)(
    result.candidates.unsqueeze(0)
)
print("同じ候補のjoint qBALD:", joint_bald_score)
assert result.candidates.shape == (2, 2)
assert joint_bald_score.shape == (1,)
assert torch.isfinite(joint_bald_score).all()


## 6. Mixed Ordinal GP：入力に設備カテゴリを含む場合

**出力のクラス0・1・2は順序付き**ですが、
**入力の設備カテゴリ0・1・2には順序がない**とします。

`cat_dims=[1]` は入力の2列目（設備ID）に
カテゴリ用カーネルを使う指定です。
`MixedOrdinalSingleTaskGPClassifier` は、
通常のOrdinal GPと同じOrdered-probit尤度・確率予測の仕組みを再利用します。


In [ ]:
mixed_X = train_X.clone()
mixed_X[:, 1] = torch.arange(mixed_X.shape[0]) % 3
mixed_quality = mixed_X[:, 0] + 0.08 * (mixed_X[:, 1] == 2).to(dtype)
mixed_Y = torch.bucketize(
    mixed_quality.contiguous(), thresholds_for_synthetic_labels
)
mixed_model = MixedOrdinalSingleTaskGPClassifier(
    mixed_X, mixed_Y, num_classes=3, cat_dims=[1], inducing_points=8
)
mixed_model.fit(epochs=4, lr=0.03)
valid_mixed_X = torch.tensor(
    [[0.2, 0.0], [0.5, 1.0], [0.8, 2.0]], dtype=dtype
)
mixed_probabilities = mixed_model.predict_proba(valid_mixed_X)
print("Mixed入力のクラス確率:\n", mixed_probabilities)
assert mixed_probabilities.shape == (3, 3)
torch.testing.assert_close(
    mixed_probabilities.sum(-1), torch.ones(3, dtype=dtype)
)


### Mixed候補の提案：合法なカテゴリと入力制約を指定する

モデル側の `cat_dims` だけでは候補最適化器はカテゴリ値を知りません。
`MixedVariableSpace` に **合法な設備ID 0・1・2** を明示します。

ここでは連続条件（第1列）が0.25以上という**既知の入力制約**も付けます。
これは「良好クラスの確率が0.25以上」などの**モデル出力制約ではありません**。


In [ ]:
mixed_bounds = torch.tensor(
    [[0.0, 0.0], [1.0, 2.0]], dtype=dtype
)
mixed_space = MixedVariableSpace(
    mixed_bounds, categorical_values={1: (0.0, 1.0, 2.0)}
)
input_constraints = CandidateConstraints(
    inequality_constraints=(
        (torch.tensor([0]), torch.tensor([1.0], dtype=dtype), 0.25),
    )
)
mixed_candidates, mixed_acq_value = optimize_acqf_sampling(
    qPredictiveEntropy(mixed_model),
    mixed_bounds,
    q=2,
    num_samples=24,
    method="sobol",
    seed=7,
    variable_space=mixed_space,
    constraints=input_constraints,
)
print("制約を満たすMixed候補:\n", mixed_candidates)
print("獲得関数値:", mixed_acq_value)
assert mixed_candidates.shape == (2, 2)
assert (mixed_candidates[:, 0] >= 0.25).all()
assert torch.isin(
    mixed_candidates[:, 1], torch.tensor([0.0, 1.0, 2.0])
).all()


### Mixed探索では制約とカテゴリを同時に守る

`optimize_acqf_sampling()` は、robotorchanが提供する**獲得関数のサンプリング最適化API**です。この例では `MixedVariableSpace` で設備カテゴリ0/1/2を指定し、`CandidateConstraints` で連続条件が0.25以上という可行領域を指定します。

候補を選んだ後は、入力制約を満たすかを必ず確認します。これはモデルのクラス確率に対する制約ではありません。

## 7. 次の実験を提案し、結果を受け取って再学習する（2ラウンド）

ここからが実際のActive Learningのサイクルです。

1. **Ask**：未観測候補だけでEntropyを計算し、測定条件を提案する
2. **Tell**：**提案された後に**初めて検査して順序ラベルを得る
3. 観測済みデータへ追加し、選んだ候補をプールから除く
4. 新しいデータでモデルを学習し直して、次の条件を提案する

実務では `torch.bucketize(...)` に相当する部分を
実際の検査結果で置き換えます。ここでは人工ラベル生成規則を使います。
未観測プールの正解ラベルは**候補選択に利用しません**。


In [ ]:
unobserved_x0 = torch.linspace(0.08, 0.92, 10, dtype=dtype)
unobserved_pool = torch.stack(
    (unobserved_x0, torch.full_like(unobserved_x0, 0.5)), dim=-1
)
observed_X = train_X.clone()
observed_Y = train_Y.clone()

for round_number in range(1, 3):
    current_model = OrdinalSingleTaskGPClassifier(
        observed_X, observed_Y, num_classes=3, inducing_points=8
    )
    current_model.fit(epochs=3, lr=0.03)
    suggested_X, suggested_value = optimize_acqf_discrete(
        acq_function=PredictiveEntropy(current_model),
        q=1,
        choices=unobserved_pool,
        unique=True,
    )
    selected_index = int(torch.where(
        torch.all(unobserved_pool == suggested_X[0], dim=-1)
    )[0][0].item())
    print(f"  Entropy獲得関数値 = {suggested_value.detach().item():.4f}")

    # ラベルは「提案後に実験で観測する」ことを人工規則で模擬します。
    observed_quality = suggested_X[:, 0] + 0.12 * suggested_X[:, 1]
    newly_observed_Y = torch.bucketize(
        observed_quality.contiguous(), thresholds_for_synthetic_labels
    )
    print(
        f"Round {round_number}: 提案={suggested_X.tolist()}, "
        f"観測ラベル={newly_observed_Y.item()}"
    )

    observed_X = torch.cat((observed_X, suggested_X), dim=0)
    observed_Y = torch.cat((observed_Y, newly_observed_Y), dim=0)
    keep = torch.ones(len(unobserved_pool), dtype=torch.bool)
    keep[selected_index] = False
    unobserved_pool = unobserved_pool[keep]

assert observed_X.shape[0] == train_X.shape[0] + 2
assert len(unobserved_pool) == 8

# 2ラウンド目で取得したラベルも反映し、全観測データで学習し直します。
final_model = OrdinalSingleTaskGPClassifier(
    observed_X, observed_Y, num_classes=3, inducing_points=8
)
final_model.fit(epochs=3, lr=0.03)


### 固定プールのAsk-and-Tellには `optimize_acqf_discrete` を使う

これまでは全候補のEntropyを計算して `argmax()` を選んでいました。ここでは同じ処理をBoTorchの公開最適化APIに任せます。

`choices=unobserved_pool` により、**未観測の合法な条件だけ**から1点を選択します。ラベルを取得した後、その点をプールから除外して再学習します。これにより同じ候補を次のラウンドで再選択しません。

## 8. 保留データで予測の読み方を確認する

実務では学習や候補選択に使わなかった保留データで検証します。
ここでは別の人工点を使い、**Accuracy・NLL・Brier score** の計算を確認します。
以下の値は少数epochの簡易例の結果であり、実用精度を示すものではありません。

- **Accuracy**：最大確率クラスが正しい比率
- **NLL**：正解クラスに割り当てた確率を評価（小さいほどよい）
- **Brier score**：確率ベクトルとOne-hot正解の二乗誤差（小さいほどよい）
- **順序の誤差**：必要ならクラス番号の絶対差も評価できますが、
  番号間隔が同じ品質差を意味するとは限りません。


In [ ]:
holdout_x0 = torch.linspace(0.1, 0.9, 12, dtype=dtype)
holdout_X = torch.stack(
    (holdout_x0, torch.full_like(holdout_x0, 0.5)), dim=-1
)
holdout_quality = holdout_X[:, 0] + 0.12 * holdout_X[:, 1]
holdout_Y = torch.bucketize(
    holdout_quality.contiguous(), thresholds_for_synthetic_labels
)
# 最終ラウンドの学習器で保留点を評価します。選択には利用しません。
holdout_proba = final_model.predict_proba(holdout_X)
accuracy = (holdout_proba.argmax(-1) == holdout_Y).double().mean()
correct_probability = holdout_proba.gather(
    -1, holdout_Y.unsqueeze(-1)
).squeeze(-1)
nll = -correct_probability.clamp_min(1e-12).log().mean()
one_hot = torch.nn.functional.one_hot(holdout_Y, num_classes=3).to(dtype)
brier = ((holdout_proba - one_hot) ** 2).sum(dim=-1).mean()
print("Accuracy:", float(accuracy))
print("NLL:", float(nll))
print("Multiclass Brier:", float(brier))
assert torch.isfinite(nll)
assert torch.isfinite(brier)


## 9. 実務での確認事項と次の学習

- **出力の順序**：0、1、2の意味と方向を先に定義し、全データで揃える
- **クラス不均衡**：3つのラベルが十分に観測されているか確認する
- **確率の校正**：AccuracyだけでなくNLL、Brier score、クラス別Recallを確認する
- **提案品質**：同じラベル予算でランダムサンプリングとも比較する
- **入力制約**：設備ID・可行領域など、実験可能な条件を探索器へ指定する
- **q-batch**：周辺平均スコアとjoint qBALDの違い、`C**q <= 256` を確認する
- **過信しない**：小さな人工例の学習成功は実際の汎化性能・安全性の保証ではない

**まとめ：** Ordinal GPは、順序ラベルを単なる数値回帰として扱うことなく、
単一の潜在GPとOrdered-probit閾値で確率をモデル化します。
学習・予測・単一点Active Learning・q-batch・Mixed入力・候補生成を
公開APIの組合せで実行できます。

次に読む：[Ordinal GPの理論](../../docs/theory/24_classification.md) /
[実装ガイド](../../docs/guide/24_classification.md) /
[分類モデルと対応範囲](../../docs/models/classification.md)


## 10. まとめ：順序分類と獲得関数最適化

| 処理 | 公開API |
|---|---|
| Ordinal分類モデル | `OrdinalSingleTaskGPClassifier` |
| Mixed Ordinal分類 | `MixedOrdinalSingleTaskGPClassifier` |
| 変分学習 | `model.fit()` |
| クラス確率と信用区間 | `predict_proba()` / `probability_posterior()` |
| 単一点の分類獲得関数 | `PredictiveEntropy` / `MarginUncertainty` / `BALD` |
| q-batch分類獲得関数 | `qPredictiveEntropy` / `qBALD` |
| 連続q=2最適化 | `SobolSearchStrategy.optimize()` |
| Mixed制約付き最適化 | `optimize_acqf_sampling()` |
| 固定プール最適化 | `optimize_acqf_discrete()` |
| ラベル取得・更新 | 観測後にデータ追加 → `fit()` |

**出力クラスの順序と、入力カテゴリの扱いは別です。** クラス0/1/2は品質の順序を表しますが、設備カテゴリ0/1/2には数値的な大小の意味を持たせません。

## Phase 4：非定常・Spectral Mixtureの比較
GibbsカーネルとSpectral Mixtureカーネルを通常入力・Mixed入力で比較します。各モデルを学習し、クラス確率を可視化して予測エントロピー最大の次候補を提案します。少数epochはAPIの動作例であり、予測性能の保証ではありません。


In [ ]:
# Phase 4: Gibbs nonstationary / Spectral Mixture — fit → probability → candidate
import torch
import matplotlib.pyplot as plt
from robotorchan.models.classification import (
    NonstationaryOrdinalGPClassifier,
    SpectralMixtureOrdinalGPClassifier,
    MixedNonstationaryOrdinalGPClassifier,
    MixedSpectralMixtureOrdinalGPClassifier,
)

torch.manual_seed(12)
x_cont = torch.linspace(-1, 1, 18, dtype=torch.double).unsqueeze(-1)
y_phase4 = torch.bucketize(
    x_cont.squeeze(-1), torch.tensor([-0.3, 0.3], dtype=torch.double)
)
grid = torch.linspace(-1, 1, 61, dtype=torch.double).unsqueeze(-1)
models_phase4 = (
    (NonstationaryOrdinalGPClassifier, False),
    (SpectralMixtureOrdinalGPClassifier, False),
    (MixedNonstationaryOrdinalGPClassifier, True),
    (MixedSpectralMixtureOrdinalGPClassifier, True),
)
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, (model_type, mixed) in zip(axes.flat, models_phase4):
    # Mixed例はカテゴリ0を固定した条件付き断面を可視化
    x = torch.cat((x_cont, (torch.arange(18) % 2).to(x_cont).unsqueeze(-1)), dim=-1) if mixed else x_cont
    candidate_x = torch.cat((grid, torch.zeros_like(grid)), dim=-1) if mixed else grid
    kwargs = {"cat_dims": [1]} if mixed else {}
    torch.manual_seed(42)  # モデルごとの乱数状態を固定し比較を再現可能にする
    if "SpectralMixture" in model_type.__name__:
        kwargs["num_mixtures"] = 2
    model = model_type(x, y_phase4, num_classes=3, **kwargs)
    model.fit(epochs=2, lr=0.001)
    with torch.no_grad():
        p = model.predict_proba(candidate_x)
        entropy = -torch.special.xlogy(p, p).sum(dim=-1)
        best_index = entropy.argmax()
        proposed = candidate_x[best_index]
    for c in range(3):
        ax.plot(grid.squeeze(-1).numpy(), p[:, c].numpy(), label=f"class {c}")
    ax.axvline(float(proposed[0]), color="black", linestyle="--", label="next candidate")
    ax.set_title(model_type.__name__)
    ax.set_ylabel("predicted probability")
    print(model_type.__name__, "candidate:", proposed.tolist())
axes[0, 0].legend(loc="best")
fig.tight_layout()
plt.show()


## Phase 5：高次元GP分類（PCA・PLS・Random Projection・ALEBO）

この節では **8次元の元入力 → 3次元の縮約入力 → fit → クラス確率 → 候補提案** を比較します。PCA・PLS・Random Projectionでは元の8次元候補をモデルへ渡します。ALEBOでは**埋め込み座標**を直接モデルへ渡します（実験空間への逆写像は別途 `ALEBOStrategy` が担当）。

ここでは学習が短時間で終わるようにepochを抑えます。PLSはラベルの数値化に依存し、特に名義尺度の多クラスではクラス番号の置換で縮約結果が変わる可能性があります。性能比較には独立した評価データが必要です。


In [ ]:
import torch
import matplotlib.pyplot as plt
from robotorchan.models.classification import (
    ALEBOOrdinalGPClassifier,
    PCAOrdinalGPClassifier,
    PLSOrdinalGPClassifier,
    RandomProjectionOrdinalGPClassifier,
)

torch.manual_seed(25)
train_X_hd = torch.rand(24, 8, dtype=torch.double)
# 3クラスの模擬観測。実際の利用では測定済みラベルに置き換えます。
train_Y_hd = torch.bucketize(train_X_hd[:, 0] + 0.5 * train_X_hd[:, 1],
                             torch.tensor([0.55, 0.95], dtype=torch.double))
candidate_X_hd = torch.rand(64, 8, dtype=torch.double)

models_hd = {
    "PCA": PCAOrdinalGPClassifier(train_X_hd, train_Y_hd, num_classes=3, n_components=3),
    "PLS": PLSOrdinalGPClassifier(train_X_hd, train_Y_hd, num_classes=3, n_components=3),
    "Random Projection": RandomProjectionOrdinalGPClassifier(
        train_X_hd, train_Y_hd, num_classes=3, n_components=3
    ),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, (name, model_hd) in zip(axes, models_hd.items()):
    model_hd.fit(epochs=2, lr=0.01)
    with torch.no_grad():
        p_hd = model_hd.predict_proba(candidate_X_hd)
        h_hd = -torch.special.xlogy(p_hd, p_hd).sum(-1)
        chosen_hd = int(h_hd.argmax())
    assert p_hd.shape == (64, 3)
    assert torch.allclose(p_hd.sum(-1), torch.ones(64, dtype=p_hd.dtype), atol=1e-5)
    ax.scatter(candidate_X_hd[:, 0].numpy(), h_hd.numpy(), s=14)
    ax.scatter(candidate_X_hd[chosen_hd, 0].item(), h_hd[chosen_hd].item(),
               marker="*", s=120, label="proposed")
    ax.set(title=name, xlabel="original input x[0]", ylabel="predictive entropy")
    ax.legend()
    print(name, "proposed original-space X:", candidate_X_hd[chosen_hd].tolist(),
          "class probabilities:", p_hd[chosen_hd].tolist())
plt.tight_layout()
plt.show()

# ALEBOモデルは元の8次元ではなく、埋め込みの3次元座標を受け取ります。
train_Z_hd = train_X_hd[:, :3].clone()
candidate_Z_hd = candidate_X_hd[:, :3].clone()
alebo_hd = ALEBOOrdinalGPClassifier(
    train_Z_hd, train_Y_hd, num_classes=3, projection=torch.eye(3, dtype=torch.double)
)
alebo_hd.fit(epochs=2, lr=0.01)
with torch.no_grad():
    alebo_p_hd = alebo_hd.predict_proba(candidate_Z_hd)
    alebo_h_hd = -torch.special.xlogy(alebo_p_hd, alebo_p_hd).sum(-1)
    alebo_idx_hd = int(alebo_h_hd.argmax())
assert alebo_p_hd.shape == (64, 3)
print("ALEBO proposed embedded Z:", candidate_Z_hd[alebo_idx_hd].tolist())
print("ALEBO class probabilities:", alebo_p_hd[alebo_idx_hd].tolist())
plt.figure(figsize=(6, 3))
plt.scatter(candidate_Z_hd[:, 0].numpy(), alebo_h_hd.numpy(), s=14)
plt.scatter(candidate_Z_hd[alebo_idx_hd, 0].item(), alebo_h_hd[alebo_idx_hd].item(),
            marker="*", s=120)
plt.xlabel("embedded coordinate z[0]")
plt.ylabel("predictive entropy")
plt.title("ALEBO embedded-space candidate")
plt.tight_layout()
plt.show()


## Phase 6：SAAS型・MAP-SAAS型の高次元GP分類

ここでは**元の8次元入力**で疎ARD事前分布を利用するモデルを比較します。
縮約（PCA等）とは異なり、入力の各次元にARD長さ尺度があり、逆長さ尺度に縮小事前分布を置きます。
両モデルとも既存の変分分類用 `fit()` を利用します。**完全Bayesian MCMCではありません。**

以下では学習、クラス確率、予測エントロピー、プール中の最大エントロピー候補提案、
散布図による確認まで実行します。学習epoch数はAPI動作確認用であり、性能を評価する値ではありません。


In [ ]:
import matplotlib.pyplot as plt
import torch

from robotorchan.models.classification import (
    MapSaasOrdinalGPClassifier,
    SaasOrdinalGPClassifier,
)

torch.manual_seed(36)
train_X_sparse = torch.rand(18, 8, dtype=torch.double)
train_Y_sparse = torch.bucketize(
    train_X_sparse[:, 0] + 0.3 * train_X_sparse[:, 1],
    torch.tensor([0.55, 0.9], dtype=torch.double),
)
candidate_X_sparse = torch.rand(64, 8, dtype=torch.double)

sparse_models = {
    "SAAS-style": SaasOrdinalGPClassifier(
        train_X_sparse, train_Y_sparse, num_classes=3, global_scale=0.1
    ),
    "MAP-SAAS-style": MapSaasOrdinalGPClassifier(
        train_X_sparse, train_Y_sparse, num_classes=3, tau=0.1
    ),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (name, sparse_model) in zip(axes, sparse_models.items()):
    sparse_model.fit(epochs=2, lr=0.005)
    with torch.no_grad():
        probabilities_sparse = sparse_model.predict_proba(candidate_X_sparse)
        entropy_sparse = -torch.special.xlogy(
            probabilities_sparse, probabilities_sparse
        ).sum(dim=-1)
        choice_sparse = int(entropy_sparse.argmax())

    assert probabilities_sparse.shape == (64, 3)
    assert torch.allclose(
        probabilities_sparse.sum(dim=-1),
        torch.ones(64, dtype=torch.double),
        atol=1e-5,
    )
    ax.scatter(candidate_X_sparse[:, 0].numpy(), entropy_sparse.numpy(), s=16)
    ax.scatter(
        candidate_X_sparse[choice_sparse, 0].item(),
        entropy_sparse[choice_sparse].item(),
        marker="*",
        s=120,
        label="proposed",
    )
    ax.set(title=name, xlabel="feature x[0]", ylabel="predictive entropy")
    ax.legend()
    print(name, "candidate:", candidate_X_sparse[choice_sparse].tolist())
    print(name, "class probability:", probabilities_sparse[choice_sparse].tolist())
plt.tight_layout()
plt.show()


### Phase 6：学習後のARD長さ尺度を確認する

SAAS型事前分布は各入力次元の**逆長さ尺度**（1 / lengthscale）を小さくする方向に働きます。値が大きい次元は、モデルの出力がその方向に敏感になり得ます。ただし逆長さ尺度は因果的な重要度ではなく、特徴量のスケールや相関にも左右されます。比較前に入力範囲を揃え、保留データの予測性能と併せて解釈します。


In [ ]:
from gpytorch.kernels import ScaleKernel

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (name, sparse_model) in zip(axes, sparse_models.items()):
    covar = sparse_model.model.covar_module
    base = covar.base_kernel if isinstance(covar, ScaleKernel) else covar
    inv_lengthscale = base.lengthscale.detach().cpu().reciprocal().reshape(-1)
    ax.bar(range(len(inv_lengthscale)), inv_lengthscale.numpy())
    ax.set(title=name, xlabel="input feature index", ylabel="inverse lengthscale")
plt.tight_layout()
plt.show()


## Phase 7：MultiTask・Block形式での分類と候補提案

2つの設備タスクで同じ3クラスを観測する例です。まずLong形式（1行が1タスクの観測）で学習し、
次に全タスクを同じ入力条件で観測するBlock形式を試します。両方ともICMのタスク間共分散を使いますが、
**厳密Kronecker専用の高速推論ではありません**。

タスク別の予測エントロピーをグラフで比較し、Block形式では各タスクのエントロピー平均が最大の入力条件を提案します。
これは実験予算を例示するプールベースの提案で、連続空間の最適化や情報重複を考慮したq-batch最適化ではありません。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.models.classification import (
    KroneckerMultiTaskOrdinalGPClassifier,
    MultiTaskOrdinalGPClassifier,
)

torch.manual_seed(47)
X_phase7 = torch.rand(15, 2, dtype=torch.double)
task0_y = torch.bucketize(
    X_phase7[:, 0] + X_phase7[:, 1] * 0.3,
    torch.tensor([0.45, 0.85], dtype=torch.double),
)
task1_y = torch.bucketize(
    X_phase7[:, 0] * 0.6 + X_phase7[:, 1] * 0.7,
    torch.tensor([0.35, 0.8], dtype=torch.double),
)
Y_phase7 = torch.stack((task0_y, task1_y), dim=-1)
task_col_phase7 = torch.arange(2, dtype=torch.double).view(1, 2, 1)
train_long_X_phase7 = torch.cat(
    (
        X_phase7[:, None, :].expand(15, 2, 2),
        task_col_phase7.expand(15, 2, 1),
    ),
    dim=-1,
).reshape(30, 3)
train_long_Y_phase7 = Y_phase7.reshape(-1)
long_model_phase7 = MultiTaskOrdinalGPClassifier(
    train_long_X_phase7, train_long_Y_phase7,
    task_feature=-1, num_classes=3, rank=2,
)
long_model_phase7.fit(epochs=2, lr=0.005)

candidate_X_phase7 = torch.rand(40, 2, dtype=torch.double)
long_candidates_phase7 = torch.cat(
    (
        candidate_X_phase7[:, None, :].expand(40, 2, 2),
        task_col_phase7.expand(40, 2, 1),
    ),
    dim=-1,
).reshape(80, 3)
with torch.no_grad():
    long_prob_phase7 = long_model_phase7.predict_proba(long_candidates_phase7)
assert long_prob_phase7.shape == (80, 3)
print("Long-format task-aware class probability:", long_prob_phase7[:2])

block_model_phase7 = KroneckerMultiTaskOrdinalGPClassifier(
    X_phase7, Y_phase7, num_classes=3, rank=2,
)
block_model_phase7.fit(epochs=2, lr=0.005)
with torch.no_grad():
    task_prob_phase7 = block_model_phase7.predict_proba(candidate_X_phase7)
    task_entropy_phase7 = -torch.special.xlogy(
        task_prob_phase7, task_prob_phase7
    ).sum(-1)
    task_mean_entropy_phase7 = task_entropy_phase7.mean(dim=-1)
    proposed_idx_phase7 = int(task_mean_entropy_phase7.argmax())
assert task_prob_phase7.shape == (40, 2, 3)
assert torch.allclose(task_prob_phase7.sum(-1), torch.ones(40, 2, dtype=torch.double), atol=1e-5)
print("Block-format candidate:", candidate_X_phase7[proposed_idx_phase7].tolist())
print("Per-task class probabilities:", task_prob_phase7[proposed_idx_phase7].tolist())

fig, ax = plt.subplots(figsize=(7, 4))
for task_id in range(2):
    ax.scatter(
        candidate_X_phase7[:, 0].numpy(),
        task_entropy_phase7[:, task_id].numpy(),
        s=18, label=f"task {task_id}",
    )
ax.scatter(
    candidate_X_phase7[proposed_idx_phase7, 0].item(),
    task_mean_entropy_phase7[proposed_idx_phase7].item(),
    s=140, marker="*", label="proposed (mean entropy)",
)
ax.set(xlabel="input x[0]", ylabel="predictive entropy", title="Phase 7: task-aware classification")
ax.legend()
plt.tight_layout()
plt.show()


### Phase 7：確率Posterior・BALDから候補点を提案する

`predict_proba()` はタスク別の `[q,m,C]`、`probability_posterior().mean` はBoTorch互換のフラットな `[q*m,C]` です。
確率Posteriorの平均と事後分散を確認し、タスク平均BALDで次候補を提案します。
**タスク平均BALDは各タスクのスコアを集約したものであり、タスク間の情報重複を厳密には評価しません。**


In [ ]:
with torch.no_grad():
    probability_posterior_phase7 = block_model_phase7.probability_posterior(
        candidate_X_phase7
    )
    probability_mean_phase7 = probability_posterior_phase7.mean.reshape(40, 2, 3)
    probability_variance_phase7 = probability_posterior_phase7.variance.reshape(40, 2, 3)
    bald_phase7 = block_model_phase7.mutual_information(
        candidate_X_phase7, num_samples=16
    )
    selected_bald_phase7 = int(bald_phase7.mean(-1).argmax())

assert probability_mean_phase7.shape == (40, 2, 3)
assert probability_variance_phase7.shape == (40, 2, 3)
assert bald_phase7.shape == (40, 2)
print("BALD proposal:", candidate_X_phase7[selected_bald_phase7].tolist())
print("Task-class probabilities:", probability_mean_phase7[selected_bald_phase7].tolist())
print("Epistemic probability variances:",
      probability_variance_phase7[selected_bald_phase7].tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for task_index in range(2):
    axes[0].scatter(
        candidate_X_phase7[:, 0].numpy(),
        bald_phase7[:, task_index].numpy(),
        s=16,
        label=f"task {task_index}",
    )
axes[0].set(xlabel="input x[0]", ylabel="BALD", title="Task-specific information gain")
axes[0].legend()
axes[1].scatter(
    candidate_X_phase7[:, 0].numpy(),
    probability_variance_phase7.mean(dim=(-1, -2)).numpy(),
    s=16,
)
axes[1].set(
    xlabel="input x[0]",
    ylabel="mean class-probability variance",
    title="Epistemic uncertainty",
)
plt.tight_layout()
plt.show()


## Multi-Fidelity GP分類：低忠実度と高忠実度の候補比較

観測の精度や計算資源を表す忠実度をモデル入力の最後の列とします。
連続入力モデルとカテゴリ設計変数を含むMixedモデルで、
学習 → クラス確率 → 忠実度別の予測エントロピー → 候補提案まで試します。
この例では忠実度の選択を自動最適化せず、高忠実度で評価する候補を提案します。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.models.classification import (
    MixedMultiFidelityOrdinalGPClassifier,
    MultiFidelityOrdinalGPClassifier,
)

torch.manual_seed(48)
n_mf = 18
train_X_mf = torch.rand(n_mf, 3, dtype=torch.double)
train_X_mf[:, 2] = (torch.arange(n_mf) % 2).to(torch.double)
train_Y_mf = torch.bucketize(
    train_X_mf[:, 0] + 0.35 * train_X_mf[:, 1] + 0.1 * train_X_mf[:, 2],
    torch.tensor([0.52, 0.95], dtype=torch.double),
)
model_mf = MultiFidelityOrdinalGPClassifier(
    train_X_mf, train_Y_mf, num_classes=3, iteration_fidelity=2
)
model_mf.fit(epochs=2, lr=0.005)

candidate_design_mf = torch.rand(32, 2, dtype=torch.double)
candidate_low_mf = torch.cat(
    (candidate_design_mf, torch.zeros(32, 1, dtype=torch.double)), dim=-1
)
candidate_high_mf = torch.cat(
    (candidate_design_mf, torch.ones(32, 1, dtype=torch.double)), dim=-1
)
with torch.no_grad():
    prob_low_mf = model_mf.predict_proba(candidate_low_mf)
    prob_high_mf = model_mf.predict_proba(candidate_high_mf)
    entropy_low_mf = -torch.special.xlogy(prob_low_mf, prob_low_mf).sum(-1)
    entropy_high_mf = -torch.special.xlogy(prob_high_mf, prob_high_mf).sum(-1)
    chosen_mf = int(entropy_high_mf.argmax())
assert prob_low_mf.shape == (32, 3)
assert prob_high_mf.shape == (32, 3)
print("Proposed high-fidelity input:", candidate_high_mf[chosen_mf].tolist())
print("High-fidelity class probabilities:", prob_high_mf[chosen_mf].tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, entropy, name in zip(
    axes,
    (entropy_low_mf, entropy_high_mf),
    ("low fidelity", "high fidelity"),
):
    ax.scatter(candidate_design_mf[:, 0].numpy(), entropy.numpy(), s=18)
    ax.set(title=name, xlabel="design x[0]", ylabel="predictive entropy")
axes[1].scatter(
    candidate_design_mf[chosen_mf, 0].item(),
    entropy_high_mf[chosen_mf].item(),
    s=140,
    marker="*",
    label="proposed",
)
axes[1].legend()
plt.tight_layout()
plt.show()

train_X_mixed_mf = train_X_mf.clone()
train_X_mixed_mf[:, 1] = (torch.arange(n_mf) % 2).to(torch.double)
mixed_model_mf = MixedMultiFidelityOrdinalGPClassifier(
    train_X_mixed_mf,
    train_Y_mf,
    num_classes=3,
    cat_dims=[1],
    iteration_fidelity=2,
)
mixed_model_mf.fit(epochs=2, lr=0.005)
mixed_candidates_mf = candidate_high_mf.clone()
mixed_candidates_mf[:, 1] = (torch.arange(32) % 2).to(torch.double)
with torch.no_grad():
    mixed_probs_mf = mixed_model_mf.predict_proba(mixed_candidates_mf)
    mixed_entropy_mf = -torch.special.xlogy(mixed_probs_mf, mixed_probs_mf).sum(-1)
    selected_mixed_mf = int(mixed_entropy_mf.argmax())
assert mixed_probs_mf.shape == (32, 3)
print("Mixed high-fidelity proposal:", mixed_candidates_mf[selected_mixed_mf].tolist())


### 既存の獲得関数による忠実度固定の候補提案

高忠実度の候補集合を用意し、既存の`PredictiveEntropy`、`qPredictiveEntropy`、`qBALD`を比較します。
`qPredictiveEntropy`は候補ごとのエントロピー平均を評価し、`qBALD`は候補間の依存を含む同時ラベル情報量をMC推定します。
いずれも忠実度の評価コストを自動的には考慮しません。


In [ ]:
from robotorchan.acquisition.active_learning.classification import (
    PredictiveEntropy,
    qBALD,
    qPredictiveEntropy,
)

with torch.no_grad():
    point_entropy_mf = PredictiveEntropy(model_mf)(candidate_high_mf.unsqueeze(-2))
    high_batches_mf = candidate_high_mf[:16].reshape(8, 2, 3)
    marginal_q_mf = qPredictiveEntropy(model_mf)(high_batches_mf)
    joint_bald_mf = qBALD(model_mf, num_samples=8)(high_batches_mf)
    proposed_single_mf = candidate_high_mf[point_entropy_mf.argmax()]
    proposed_q_mf = high_batches_mf[joint_bald_mf.argmax()]
assert point_entropy_mf.shape == (32,)
assert marginal_q_mf.shape == (8,)
assert joint_bald_mf.shape == (8,)
assert proposed_q_mf.shape == (2, 3)
print("Entropy proposal:", proposed_single_mf.tolist())
print("qBALD proposal:", proposed_q_mf.tolist())


## DeepGP分類：確率サンプル・潜在不確実性・候補提案

DeepGPの潜在分布は一般にガウスではありません。学習後の潜在事後サンプルから、クラス確率の平均と分散を確認します。既存のPredictiveEntropyを用いて次の候補点を提案し、結果を図示します。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.acquisition.active_learning.classification import PredictiveEntropy
from robotorchan.models.classification import OrdinalSingleTaskDeepGPClassifier

torch.manual_seed(61)
train_X_dgp = torch.rand(12, 2, dtype=torch.double)
train_Y_dgp = torch.bucketize(
    train_X_dgp[:, 0],
    torch.tensor([0.35, 0.7], dtype=torch.double),
)
model_dgp = OrdinalSingleTaskDeepGPClassifier(
    train_X_dgp, train_Y_dgp,
    num_classes=3,
    hidden_dims=(2,),
    num_inducing=5,
    posterior_samples=12,
)
model_dgp.fit(epochs=2, lr=0.005, num_likelihood_samples=4)

candidate_X_dgp = torch.stack(
    (torch.linspace(0.0, 1.0, 30, dtype=torch.double),
     torch.full((30,), 0.5, dtype=torch.double)),
    dim=-1,
)
model_dgp.eval()
with torch.no_grad():
    posterior_dgp = model_dgp.posterior(candidate_X_dgp)
    probability_posterior_dgp = model_dgp.probability_posterior(candidate_X_dgp)
    probability_mean_dgp = probability_posterior_dgp.mean
    epistemic_var_dgp = probability_posterior_dgp.variance
    entropy_acq_dgp = PredictiveEntropy(model_dgp)
    entropy_scores_dgp = entropy_acq_dgp(candidate_X_dgp.unsqueeze(-2))
    selected_dgp = int(entropy_scores_dgp.argmax())

assert posterior_dgp.mean.shape[-2:] == (30, 1)
assert probability_mean_dgp.shape == (30, 3)
assert epistemic_var_dgp.shape == (30, 3)
assert entropy_scores_dgp.shape == (30,)
print("DeepGP candidate:", candidate_X_dgp[selected_dgp].tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for class_id in range(3):
    axes[0].plot(
        candidate_X_dgp[:, 0].numpy(),
        probability_mean_dgp[:, class_id].numpy(),
        label=f"class {class_id}",
    )
axes[0].set(xlabel="x[0]", ylabel="predictive class probability",
            title="DeepGP predictive probabilities")
axes[0].legend()
axes[1].plot(candidate_X_dgp[:, 0].numpy(), entropy_scores_dgp.numpy())
axes[1].plot(
    candidate_X_dgp[selected_dgp, 0].item(),
    entropy_scores_dgp[selected_dgp].item(),
    "*", markersize=13,
)
axes[1].set(xlabel="x[0]", ylabel="predictive entropy",
            title="Active learning proposal")
plt.tight_layout()
plt.show()


## InfiniteWidthBNN：無限幅ReLUカーネルによる分類

連続入力とMixed入力の両方で、既存のInfiniteWidthReLUKernelを分類用の潜在GPに適用します。学習→確率予測→確率の不確実性→既存獲得関数での候補提案までを実行します。DeepGPと異なり複数の確率的なGP層は使用せず、単一GPの共分散を変更しています。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.acquisition.active_learning.classification import PredictiveEntropy
from robotorchan.models.classification import (
    InfiniteWidthBNNOrdinalGPClassifier,
    MixedInfiniteWidthBNNOrdinalGPClassifier,
)

torch.manual_seed(72)
train_X_nngp = torch.rand(15, 2, dtype=torch.double)
train_Y_nngp = torch.bucketize(
    train_X_nngp[:, 0] + 0.15 * train_X_nngp[:, 1],
    torch.tensor([0.38, 0.76], dtype=torch.double),
)
model_nngp = InfiniteWidthBNNOrdinalGPClassifier(
    train_X_nngp, train_Y_nngp, num_classes=3, depth=2
)
model_nngp.fit(epochs=2, lr=0.005)

candidate_X_nngp = torch.stack(
    (
        torch.linspace(0.0, 1.0, 32, dtype=torch.double),
        torch.full((32,), 0.5, dtype=torch.double),
    ),
    dim=-1,
)
with torch.no_grad():
    probability_nngp = model_nngp.predict_proba(candidate_X_nngp)
    posterior_nngp = model_nngp.probability_posterior(
        candidate_X_nngp
    )
    entropy_nngp = PredictiveEntropy(model_nngp)(
        candidate_X_nngp.unsqueeze(-2)
    )
    proposed_nngp = int(entropy_nngp.argmax())

assert probability_nngp.shape == (32, 3)
assert posterior_nngp.mean.shape == (32, 3)
assert posterior_nngp.variance.shape == (32, 3)
assert entropy_nngp.shape == (32,)
print("NNGP candidate:", candidate_X_nngp[proposed_nngp].tolist())

fig, ax = plt.subplots(figsize=(8, 3.5))
for class_id_nngp in range(3):
    ax.plot(
        candidate_X_nngp[:, 0].numpy(),
        probability_nngp[:, class_id_nngp].numpy(),
        label=f"class {class_id_nngp}",
    )
ax.axvline(candidate_X_nngp[proposed_nngp, 0].item(), linestyle="--", label="proposed")
ax.set(xlabel="design x[0]", ylabel="class probability", title="Infinite-width NNGP classification")
ax.legend()
plt.tight_layout()
plt.show()

train_X_mixed_nngp = train_X_nngp.clone()
train_X_mixed_nngp[:, 1] = (torch.arange(15) % 2).to(torch.double)
mixed_model_nngp = MixedInfiniteWidthBNNOrdinalGPClassifier(
    train_X_mixed_nngp, train_Y_nngp, num_classes=3, cat_dims=[1], depth=2
)
mixed_model_nngp.fit(epochs=2, lr=0.005)
candidate_X_mixed_nngp = candidate_X_nngp.clone()
candidate_X_mixed_nngp[:, 1] = (torch.arange(32) % 2).to(torch.double)
with torch.no_grad():
    mixed_prob_nngp = mixed_model_nngp.predict_proba(candidate_X_mixed_nngp)
    mixed_scores_nngp = PredictiveEntropy(mixed_model_nngp)(
        candidate_X_mixed_nngp.unsqueeze(-2)
    )
    proposed_mixed_nngp = int(mixed_scores_nngp.argmax())
assert mixed_prob_nngp.shape == (32, 3)
print("Mixed NNGP proposal:", candidate_X_mixed_nngp[proposed_mixed_nngp].tolist())


## Phase 10: ContextualとHierarchical GP分類

分類結果に影響する実験条件（context）と、製造ライン・装置などの階層（group / subgroup）を識別子として渡します。ここでは既知の条件のもとでモデルを学習し、クラス確率の変化を可視化したうえで、既存の獲得関数を使用して候補を選びます。コンテキストIDは最適化する連続変数ではありません。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.acquisition.active_learning.classification import PredictiveEntropy
from robotorchan.models.classification import (
    ContextualOrdinalGPClassifier,
    HierarchicalOrdinalGPClassifier,
)

torch.manual_seed(107)
n_ctx_p10 = 18
design_ctx_p10 = torch.rand(n_ctx_p10, 1, dtype=torch.double)
context_id_p10 = (torch.arange(n_ctx_p10) % 2).to(torch.double).unsqueeze(-1)
train_X_ctx_p10 = torch.cat((design_ctx_p10, context_id_p10), dim=-1)
train_Y_ctx_p10 = torch.bucketize(
    design_ctx_p10[:, 0] + 0.15 * context_id_p10[:, 0],
    torch.tensor([0.38, 0.75], dtype=torch.double),
)
context_model_p10 = ContextualOrdinalGPClassifier(
    train_X_ctx_p10, train_Y_ctx_p10, context_feature=1, num_classes=3
)
context_model_p10.fit(epochs=2, lr=0.005)
grid_ctx_p10 = torch.linspace(0, 1, 30, dtype=torch.double).unsqueeze(-1)
fig, ax = plt.subplots(figsize=(8, 3.5))
with torch.no_grad():
    for ctx_p10 in (0, 1):
        candidate_ctx_p10 = torch.cat(
            (grid_ctx_p10, torch.full_like(grid_ctx_p10, ctx_p10)), dim=-1
        )
        probabilities_ctx_p10 = context_model_p10.predict_proba(candidate_ctx_p10)
        ax.plot(
            grid_ctx_p10[:, 0].numpy(),
            probabilities_ctx_p10[:, 2].numpy(),
            label=f"context {ctx_p10}: P(class 2)",
        )
    score_ctx_p10 = PredictiveEntropy(context_model_p10)(
        candidate_ctx_p10.unsqueeze(-2)
    )
    proposal_ctx_p10 = candidate_ctx_p10[int(score_ctx_p10.argmax())]
assert probabilities_ctx_p10.shape == (30, 3)
assert score_ctx_p10.shape == (30,)
print("Contextual candidate:", proposal_ctx_p10.tolist())
ax.set(xlabel="design x", ylabel="P(class 2)", title="Context-dependent predictions")
ax.legend()
plt.tight_layout()
plt.show()

n_hier_p10 = 16
design_hier_p10 = torch.rand(n_hier_p10, 1, dtype=torch.double)
group_hier_p10 = (torch.arange(n_hier_p10) // 8).to(torch.double).unsqueeze(-1)
subgroup_hier_p10 = (torch.arange(n_hier_p10) // 4).to(torch.double).unsqueeze(-1)
train_X_hier_p10 = torch.cat(
    (design_hier_p10, group_hier_p10, subgroup_hier_p10), dim=-1
)
train_Y_hier_p10 = torch.bucketize(
    design_hier_p10[:, 0] + 0.08 * subgroup_hier_p10[:, 0],
    torch.tensor([0.38, 0.78], dtype=torch.double),
)
hier_model_p10 = HierarchicalOrdinalGPClassifier(
    train_X_hier_p10,
    train_Y_hier_p10,
    group_feature=1,
    subgroup_feature=2,
    num_classes=3,
)
hier_model_p10.fit(epochs=2, lr=0.005)
candidate_hier_p10 = torch.cat(
    (
        grid_ctx_p10,
        torch.ones_like(grid_ctx_p10),
        torch.full_like(grid_ctx_p10, 3.0),
    ),
    dim=-1,
)
with torch.no_grad():
    hierarchical_probs_p10 = hier_model_p10.predict_proba(candidate_hier_p10)
    hierarchical_scores_p10 = PredictiveEntropy(hier_model_p10)(
        candidate_hier_p10.unsqueeze(-2)
    )
    hierarchical_choice_p10 = candidate_hier_p10[
        int(hierarchical_scores_p10.argmax())
    ]
assert hierarchical_probs_p10.shape == (30, 3)
assert hierarchical_scores_p10.shape == (30,)
print("Hierarchical candidate:", hierarchical_choice_p10.tolist())

fig, ax = plt.subplots(figsize=(8, 3.5))
for class_id_p10 in range(3):
    ax.plot(
        grid_ctx_p10[:, 0].numpy(),
        hierarchical_probs_p10[:, class_id_p10].numpy(),
        label=f"class {class_id_p10}",
    )
ax.axvline(
    hierarchical_choice_p10[0].item(), linestyle="--", label="proposed"
)
ax.set(xlabel="design x", ylabel="class probability",
       title="Group 1 / subgroup 3")
ax.legend()
plt.tight_layout()
plt.show()


## ラベルノイズ：真のクラス確率と観測ラベル確率

観測ラベルに一定割合の誤記録があると仮定し、既知のクラス遷移行列をモデルに指定して学習します。真のクラス確率（clean）と観測されるラベルの確率（observed）をグラフで比較し、既存のPredictiveEntropyで次の候補を提案します。ここでの獲得関数は観測ラベルの予測不確実性を対象とし、真のクラスの不確実性とは必ずしも一致しません。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.acquisition.active_learning.classification import PredictiveEntropy
from robotorchan.models.classification import LabelNoiseOrdinalGPClassifier

torch.manual_seed(202)
X_noise = torch.linspace(0.0, 1.0, 18, dtype=torch.double).unsqueeze(-1)
y_noise = torch.bucketize(X_noise[:, 0], torch.tensor([0.35, 0.7], dtype=torch.double))
transition_noise = torch.tensor(
    [[0.85, 0.15, 0.0], [0.05, 0.85, 0.10], [0.0, 0.2, 0.8]],
    dtype=torch.double,
)
model_noise = LabelNoiseOrdinalGPClassifier(
    X_noise, y_noise, num_classes=3, transition_matrix=transition_noise
)
model_noise.fit(epochs=2, lr=0.005)
candidate_noise = torch.linspace(0.0, 1.0, 32, dtype=torch.double).unsqueeze(-1)
with torch.no_grad():
    torch.manual_seed(53)
    clean_noise = model_noise.predict_clean_proba(candidate_noise)
    torch.manual_seed(53)
    observed_noise = model_noise.predict_proba(candidate_noise)
    probability_noise = model_noise.probability_posterior(
        candidate_noise
    )
    entropy_noise = PredictiveEntropy(model_noise)(candidate_noise.unsqueeze(-2))
    proposed_noise = int(entropy_noise.argmax())

assert clean_noise.shape == observed_noise.shape == (32, 3)
assert probability_noise.mean.shape == probability_noise.variance.shape == (32, 3)
assert entropy_noise.shape == (32,)
torch.testing.assert_close(
    observed_noise, clean_noise @ transition_noise, atol=1e-5, rtol=1e-5
)
print("Proposed candidate:", candidate_noise[proposed_noise].tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for class_index_noise in range(3):
    axes[0].plot(
        candidate_noise[:, 0].numpy(),
        clean_noise[:, class_index_noise].numpy(),
        label=f"clean class {class_index_noise}",
    )
    axes[0].plot(
        candidate_noise[:, 0].numpy(),
        observed_noise[:, class_index_noise].numpy(),
        linestyle="--",
        label=f"observed class {class_index_noise}",
    )
axes[0].set(xlabel="design x", ylabel="class probability", title="Clean vs observed labels")
axes[0].legend(ncol=2, fontsize=8)
axes[1].plot(candidate_noise[:, 0].numpy(), entropy_noise.numpy())
axes[1].axvline(
    candidate_noise[proposed_noise, 0].item(), linestyle="--", label="proposed"
)
axes[1].set(xlabel="design x", ylabel="predictive entropy", title="Next label proposal")
axes[1].legend()
plt.tight_layout()
plt.show()


## 入力不確実性：連続摂動とカテゴリ候補分布

次の例では、候補点を指定どおりに実現できない場合のクラス確率を求めます。連続入力ではガウス摂動のモンテカルロ積分、カテゴリ入力では有限サポートの重み付き和を使います。基準となる既存のPredictiveEntropyも示しますが、**入力不確実性込みの候補提案**には、周辺化した確率からエントロピーを計算します。学習入力にノイズを加える実装ではありません。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.acquisition.active_learning.classification import PredictiveEntropy
from robotorchan.models.classification import (
    ContinuousUncertainInputOrdinalGPClassifier,
    UncertainCategoricalOrdinalGPClassifier,
)

torch.manual_seed(120)
train_X_u12 = torch.linspace(0.0, 1.0, 15, dtype=torch.double).unsqueeze(-1)
train_Y_u12 = torch.bucketize(
    train_X_u12[:, 0], torch.tensor([0.33, 0.7], dtype=torch.double)
)
continuous_model_u12 = ContinuousUncertainInputOrdinalGPClassifier(
    train_X_u12, train_Y_u12, num_classes=3
)
continuous_model_u12.fit(epochs=2, lr=0.005)

candidate_u12 = torch.linspace(0, 1, 24, dtype=torch.double).unsqueeze(-1)
with torch.no_grad():
    nominal_probability_u12 = continuous_model_u12.predict_proba(candidate_u12)
    uncertain_probability_u12 = continuous_model_u12.predict_proba_with_input_uncertainty(
        candidate_u12,
        input_std=torch.full_like(candidate_u12, 0.06),
        num_samples=12,
        generator=torch.Generator().manual_seed(18),
    )
    nominal_scores_u12 = PredictiveEntropy(continuous_model_u12)(
        candidate_u12.unsqueeze(-2)
    )
    uncertain_scores_u12 = -torch.special.xlogy(
        uncertain_probability_u12, uncertain_probability_u12
    ).sum(dim=-1)
    next_candidate_u12 = candidate_u12[int(uncertain_scores_u12.argmax())]

assert uncertain_probability_u12.shape == (24, 3)
assert uncertain_scores_u12.shape == nominal_scores_u12.shape == (24,)
print("Continuous uncertainty candidate:", next_candidate_u12.tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for label_u12 in range(3):
    axes[0].plot(
        candidate_u12[:, 0].numpy(),
        nominal_probability_u12[:, label_u12].numpy(),
        label=f"nominal class {label_u12}",
    )
    axes[0].plot(
        candidate_u12[:, 0].numpy(),
        uncertain_probability_u12[:, label_u12].numpy(),
        linestyle="--",
        label=f"uncertain class {label_u12}",
    )
axes[0].legend(ncol=2, fontsize=8)
axes[0].set(xlabel="design x", ylabel="class probability", title="Input perturbations")
axes[1].plot(candidate_u12[:, 0].numpy(), nominal_scores_u12.numpy(), label="nominal")
axes[1].plot(candidate_u12[:, 0].numpy(), uncertain_scores_u12.numpy(), label="uncertain")
axes[1].axvline(next_candidate_u12[0].item(), linestyle="--", label="proposed")
axes[1].set(xlabel="design x", ylabel="predictive entropy", title="Candidate proposal")
axes[1].legend()
plt.tight_layout()
plt.show()

train_X_cat_u12 = torch.cat(
    (
        train_X_u12,
        (torch.arange(15) % 2).to(torch.double).unsqueeze(-1),
    ),
    dim=-1,
)
cat_model_u12 = UncertainCategoricalOrdinalGPClassifier(
    train_X_cat_u12,
    train_Y_u12,
    num_classes=3,
    cat_dims=[1],
    uncertain_cat_dim=1,
)
cat_model_u12.fit(epochs=2, lr=0.005)

candidate_cat_u12 = torch.cat(
    (candidate_u12, torch.zeros_like(candidate_u12)), dim=-1
)
cat_values_u12 = torch.tensor([0.0, 1.0], dtype=torch.double)
cat_weights_u12 = torch.tensor([[0.4, 0.6]], dtype=torch.double).expand(24, 2)
with torch.no_grad():
    marginalized_cat_u12 = cat_model_u12.predict_proba_with_category_uncertainty(
        candidate_cat_u12,
        category_values=cat_values_u12,
        category_probabilities=cat_weights_u12,
    )
    cat_entropy_u12 = -torch.special.xlogy(
        marginalized_cat_u12, marginalized_cat_u12
    ).sum(dim=-1)
    next_cat_u12 = candidate_cat_u12[int(cat_entropy_u12.argmax())]
assert marginalized_cat_u12.shape == (24, 3)
print("Categorical uncertainty candidate (nominal code 0):", next_cat_u12.tolist())

fig, ax = plt.subplots(figsize=(8, 3.5))
for label_u12 in range(3):
    ax.plot(
        candidate_cat_u12[:, 0].numpy(),
        marginalized_cat_u12[:, label_u12].numpy(),
        label=f"class {label_u12}",
    )
ax.axvline(next_cat_u12[0].item(), linestyle="--", label="proposed")
ax.set(
    xlabel="design x",
    ylabel="marginalized class probability",
    title="Uncertain categorical input",
)
ax.legend()
plt.tight_layout()
plt.show()


## Ensemble・非GP分類と候補点提案

scikit-learnのRandom ForestとExtra Treesを学習し、個別モデルのクラス確率、Ensemble平均・モデル間分散・獲得関数を可視化します。通常の分類に加え、GPを1つEnsembleに加えた混合モデルの候補提案まで確認します。単一の非GP分類器には潜在GP Posteriorはありません。ここでの非GP順序ラベル分類はOrdered Probitを学習しません。


In [ ]:
import matplotlib.pyplot as plt
import torch
from robotorchan.acquisition.active_learning.classification import PredictiveEntropy
from robotorchan.models.classification import (
    ExtraTreesOrdinalClassifier,
    OrdinalClassificationEnsemble,
    RandomForestOrdinalClassifier,
    OrdinalSingleTaskGPClassifier,
)

torch.manual_seed(113)
train_x_e13 = torch.linspace(0, 1, 24, dtype=torch.double).unsqueeze(-1)
train_y_e13 = torch.bucketize(
    train_x_e13[:, 0] + 0.08 * torch.sin(12 * train_x_e13[:, 0]),
    torch.tensor([0.32, 0.68], dtype=torch.double),
)
rf_e13 = RandomForestOrdinalClassifier(
    train_x_e13, train_y_e13, num_classes=3, n_estimators=20, random_state=1
)
extra_e13 = ExtraTreesOrdinalClassifier(
    train_x_e13, train_y_e13, num_classes=3, n_estimators=20, random_state=2
)
rf_e13.fit()
extra_e13.fit()
ensemble_e13 = OrdinalClassificationEnsemble(
    rf_e13, extra_e13, weights=torch.tensor([0.4, 0.6], dtype=torch.double)
)
candidate_x_e13 = torch.linspace(0, 1, 40, dtype=torch.double).unsqueeze(-1)
with torch.no_grad():
    rf_probs_e13 = rf_e13.predict_proba(candidate_x_e13)
    mean_probs_e13 = ensemble_e13.predict_proba(candidate_x_e13)
    empirical_posterior_e13 = ensemble_e13.probability_posterior(candidate_x_e13)
    disagreement_e13 = ensemble_e13.mutual_information(candidate_x_e13)
    entropy_e13 = PredictiveEntropy(ensemble_e13)(candidate_x_e13.unsqueeze(-2))
    chosen_e13 = candidate_x_e13[int(entropy_e13.argmax())]
assert mean_probs_e13.shape == (40, 3)
assert empirical_posterior_e13.variance.shape == (40, 3)
print("Non-GP ensemble proposal:", chosen_e13.tolist())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ci_e13 in range(3):
    axes[0].plot(candidate_x_e13[:, 0].numpy(), mean_probs_e13[:, ci_e13].numpy(),
                 label=f"class {ci_e13}")
axes[0].set(xlabel="x", ylabel="class probability", title="Ensemble class probabilities")
axes[0].legend()
axes[1].plot(candidate_x_e13[:, 0].numpy(), entropy_e13.numpy(), label="predictive entropy")
axes[1].plot(candidate_x_e13[:, 0].numpy(), disagreement_e13.numpy(),
             label="between-member disagreement")
axes[1].axvline(chosen_e13[0].item(), linestyle="--", label="proposal")
axes[1].set(xlabel="x", ylabel="score", title="Uncertainty and candidate")
axes[1].legend()
plt.tight_layout()
plt.show()

gp_e13 = OrdinalSingleTaskGPClassifier(train_x_e13, train_y_e13, num_classes=3)
gp_e13.fit(epochs=2, lr=0.005)
hybrid_e13 = OrdinalClassificationEnsemble(
    rf_e13, gp_e13, weights=torch.tensor([0.5, 0.5], dtype=torch.double)
)
with torch.no_grad():
    torch.manual_seed(17)
    hybrid_probs_e13 = hybrid_e13.predict_proba(candidate_x_e13)
    torch.manual_seed(17)
    hybrid_scores_e13 = PredictiveEntropy(hybrid_e13)(candidate_x_e13.unsqueeze(-2))
    hybrid_candidate_e13 = candidate_x_e13[int(hybrid_scores_e13.argmax())]
assert hybrid_probs_e13.shape == (40, 3)
print("GP + non-GP ensemble proposal:", hybrid_candidate_e13.tolist())
